In [5]:
import random
import numpy as np
import json
import pickle
import numpy as np

import nltk
from nltk.stem import WordNetLemmatizer
from keras.models import load_model
import pickle

In [6]:
lematizer = WordNetLemmatizer()
intents = json.loads(open("../intents.json").read())

words = pickle.load(open("../training_outputs/words.pkl", "rb"))
classes = pickle.load(open("../training_outputs/classes.pkl", "rb"))
model = load_model("../model/chatbot_model.h5")


In [7]:

# cleaning up the sentences
def clean_up_sentence(sentence):
    # Tokenize sentence
    sentence_words = nltk.word_tokenize(sentence)
    # Lemmatize the sentence
    sentence_words = [lematizer.lemmatize(word) for word in sentence_words]
    return sentence_words


# Convert a sentence into a bag of words (0s and 1s)
def bag_of_words(sentence):
    sentence_words = clean_up_sentence(sentence)
    bag = [0] * len(words)

    for w in sentence_words:
        for i, word in enumerate(words):
            if word == w:
                bag[i] = 1
    return np.array(bag)


def predict_class(sentence):
    bow = bag_of_words(sentence)
    result = model.predict(np.array([bow]))[0]
    # Because we used the softmax, we have a likelihood probability next to the parameters. Using 25% as a threshold...
    Error_thresh = 0.25
    results = [[i, r] for i, r in enumerate(result) if r > Error_thresh]
    results.sort(key=lambda x: x[1], reverse=True)
    return_list = []
    for r in results:
        return_list.append({"intent": classes[r[0]], "probability": str(r[1])})
        return return_list


def get_response(intents_list, intents_json):
    tag = intents_list[0]['intent']
    list_of_intents = intents_json['intents']
    for i in list_of_intents:
        if i['tag']== tag:
            result = random.choice(i['responses'])
            break
    return result

In [8]:
message = input()
ints = predict_class(message)
res = get_response(ints, intents)
print(res)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
Hey!
